# Module 43: Production Serving Patterns

Interactive exploration of batched inference, dynamic batching,
torch.compile + CUDA Graphs, and monitoring for production model serving.

In [ ]:
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Batched Inference: Padding & Masking

Variable-length inputs padded to batch max with attention masks.

In [ ]:
def pad_sequences(sequences, pad_value=0):
    lengths = [s.size(0) for s in sequences]
    max_len = max(lengths)
    batch_size = len(sequences)
    
    padded = torch.full((batch_size, max_len), pad_value, dtype=sequences[0].dtype)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    
    for i, (seq, length) in enumerate(zip(sequences, lengths)):
        padded[i, :length] = seq
        mask[i, :length] = True
    
    return padded, mask, lengths

# Create variable-length sequences
sequences = [torch.randint(0, 1000, (l,)) for l in [5, 12, 3, 8, 20, 7]]
padded, mask, lengths = pad_sequences(sequences)

print(f"Input lengths: {lengths}")
print(f"Padded shape: {padded.shape}")
print(f"Mask shape: {mask.shape}")
print(f"\nMask visualization (1=real, 0=pad):")
for i in range(len(sequences)):
    bars = '█' * lengths[i] + '░' * (padded.size(1) - lengths[i])
    print(f"  seq[{i}] len={lengths[i]:2d}: {bars}")

## 2. Throughput vs Batch Size

Measure how throughput scales with batch size.

In [ ]:
model = nn.Sequential(
    nn.Linear(128, 512),
    nn.GELU(),
    nn.Linear(512, 512),
    nn.GELU(),
    nn.Linear(512, 64),
).to(device).eval()

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
throughputs = []
latencies = []

for bs in batch_sizes:
    x = torch.randn(bs, 128, device=device)
    
    # Warmup
    for _ in range(10):
        with torch.no_grad():
            _ = model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Measure
    n_iter = 100
    start = time.perf_counter()
    for _ in range(n_iter):
        with torch.no_grad():
            _ = model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    
    samples_per_sec = (bs * n_iter) / elapsed
    latency_ms = (elapsed / n_iter) * 1000
    throughputs.append(samples_per_sec)
    latencies.append(latency_ms)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(batch_sizes, throughputs, 'b-o')
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Throughput (samples/sec)')
ax1.set_title('Throughput vs Batch Size')
ax1.set_xscale('log', base=2)
ax1.grid(True, alpha=0.3)

ax2.plot(batch_sizes, latencies, 'r-o')
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Batch Latency (ms)')
ax2.set_title('Latency vs Batch Size')
ax2.set_xscale('log', base=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBatch size 1 throughput: {throughputs[0]:.0f} samples/sec")
print(f"Batch size {batch_sizes[-1]} throughput: {throughputs[-1]:.0f} samples/sec")
print(f"Speedup: {throughputs[-1]/throughputs[0]:.1f}x")

## 3. torch.compile Speedup

Compare eager vs compiled model latency.

In [ ]:
eager_model = nn.Sequential(
    nn.Linear(256, 512),
    nn.GELU(),
    nn.Linear(512, 512),
    nn.GELU(),
    nn.Linear(512, 256),
).to(device).eval()

compiled_model = torch.compile(eager_model, mode='reduce-overhead')

x = torch.randn(16, 256, device=device)

# Warmup compiled model
print("Warming up compiled model...")
for _ in range(20):
    with torch.no_grad():
        _ = compiled_model(x)
if device.type == 'cuda':
    torch.cuda.synchronize()
print("Warmup done.")

def measure_latency(model_fn, x, n=200):
    times = []
    for _ in range(n):
        start = time.perf_counter()
        with torch.no_grad():
            _ = model_fn(x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)
    return sorted(times)

eager_times = measure_latency(eager_model, x)
compiled_times = measure_latency(compiled_model, x)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(eager_times, bins=30, alpha=0.6, label=f'Eager (p50={eager_times[100]:.3f}ms)', color='red')
ax.hist(compiled_times, bins=30, alpha=0.6, label=f'Compiled (p50={compiled_times[100]:.3f}ms)', color='blue')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('Inference Latency Distribution: Eager vs Compiled')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

speedup = eager_times[100] / compiled_times[100]
print(f"\nSpeedup (p50): {speedup:.2f}x")
print(f"Eager p95: {eager_times[190]:.3f}ms")
print(f"Compiled p95: {compiled_times[190]:.3f}ms")

## 4. Dynamic Batching Simulation

Simulate request arrivals and measure batching efficiency.

In [ ]:
import random

def simulate_dynamic_batching(
    n_requests=200,
    arrival_rate_ms=5.0,
    max_batch_size=16,
    max_wait_ms=25.0,
):
    """Simulate request arrivals and dynamic batch formation."""
    random.seed(42)
    
    # Generate arrival times (Poisson process)
    arrivals = []
    t = 0
    for _ in range(n_requests):
        t += random.expovariate(1.0 / arrival_rate_ms)
        arrivals.append(t)
    
    batches = []
    current_batch = []
    batch_start = arrivals[0]
    
    for arrival in arrivals:
        current_batch.append(arrival)
        
        time_since_start = arrival - batch_start
        if len(current_batch) >= max_batch_size or time_since_start >= max_wait_ms:
            batches.append({
                'size': len(current_batch),
                'wait_ms': time_since_start,
                'trigger': 'size' if len(current_batch) >= max_batch_size else 'timeout',
            })
            current_batch = []
            batch_start = arrival
    
    if current_batch:
        batches.append({
            'size': len(current_batch),
            'wait_ms': arrivals[-1] - batch_start,
            'trigger': 'flush',
        })
    
    return batches

# Run simulation with different configs
configs = [
    {'max_batch_size': 8, 'max_wait_ms': 10},
    {'max_batch_size': 16, 'max_wait_ms': 25},
    {'max_batch_size': 32, 'max_wait_ms': 50},
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, cfg in zip(axes, configs):
    batches = simulate_dynamic_batching(**cfg)
    sizes = [b['size'] for b in batches]
    
    ax.hist(sizes, bins=range(1, cfg['max_batch_size'] + 2), alpha=0.7, edgecolor='black')
    ax.axvline(sum(sizes)/len(sizes), color='red', linestyle='--', label=f'avg={sum(sizes)/len(sizes):.1f}')
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Count')
    ax.set_title(f"max_bs={cfg['max_batch_size']}, wait={cfg['max_wait_ms']}ms")
    ax.legend()
    
    size_triggers = sum(1 for b in batches if b['trigger'] == 'size')
    timeout_triggers = sum(1 for b in batches if b['trigger'] == 'timeout')
    ax.text(0.95, 0.85, f'size: {size_triggers}\ntimeout: {timeout_triggers}',
            transform=ax.transAxes, ha='right', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Dynamic Batching: Batch Size Distribution (arrival_rate=5ms)', y=1.02)
plt.tight_layout()
plt.show()

## 5. Monitoring: Latency Histogram

Track and visualize serving latency distribution.

In [ ]:
# Simulate realistic latency distribution
random.seed(123)
n_requests = 1000

# Mix of fast (normal serving) and slow (cold start / GC) requests
latencies = []
for _ in range(n_requests):
    if random.random() < 0.95:  # 95% normal
        latencies.append(random.gauss(5.0, 1.5))
    elif random.random() < 0.8:  # 4% slightly slow
        latencies.append(random.gauss(15.0, 3.0))
    else:  # 1% tail latency spike
        latencies.append(random.gauss(50.0, 10.0))

latencies = [max(0.5, l) for l in latencies]
latencies.sort()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(latencies, bins=50, alpha=0.7, edgecolor='black', color='steelblue')
p50 = latencies[500]
p95 = latencies[950]
p99 = latencies[990]
ax1.axvline(p50, color='green', linestyle='--', label=f'p50={p50:.1f}ms')
ax1.axvline(p95, color='orange', linestyle='--', label=f'p95={p95:.1f}ms')
ax1.axvline(p99, color='red', linestyle='--', label=f'p99={p99:.1f}ms')
ax1.set_xlabel('Latency (ms)')
ax1.set_ylabel('Count')
ax1.set_title('Latency Distribution (n=1000)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# CDF
percentiles = [i/len(latencies)*100 for i in range(len(latencies))]
ax2.plot(latencies, percentiles, 'b-', linewidth=1.5)
ax2.axhline(50, color='green', linestyle=':', alpha=0.5)
ax2.axhline(95, color='orange', linestyle=':', alpha=0.5)
ax2.axhline(99, color='red', linestyle=':', alpha=0.5)
ax2.set_xlabel('Latency (ms)')
ax2.set_ylabel('Percentile')
ax2.set_title('Latency CDF')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, max(latencies) * 1.1)

plt.tight_layout()
plt.show()

print(f"SLO check (target p99 < 100ms): {'PASS ✓' if p99 < 100 else 'FAIL ✗'}")
print(f"  p50={p50:.2f}ms, p95={p95:.2f}ms, p99={p99:.2f}ms")

## 6. Exercise: Queue Depth vs Latency Tradeoff

**Task:** Simulate how different `max_wait_ms` values affect:
1. Average batch size (higher = better throughput)
2. Request wait time (higher = worse latency)

Plot the tradeoff curve.

In [ ]:
# Exercise: vary max_wait_ms and plot avg_batch_size vs avg_wait_time
wait_values = [5, 10, 15, 20, 30, 50, 75, 100]
avg_sizes = []
avg_waits = []

for wait_ms in wait_values:
    batches = simulate_dynamic_batching(
        n_requests=500,
        arrival_rate_ms=8.0,
        max_batch_size=32,
        max_wait_ms=wait_ms,
    )
    sizes = [b['size'] for b in batches]
    waits = [b['wait_ms'] for b in batches]
    avg_sizes.append(sum(sizes) / len(sizes))
    avg_waits.append(sum(waits) / len(waits))

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(avg_waits, avg_sizes, c=wait_values, cmap='viridis', s=100, zorder=5)
ax.plot(avg_waits, avg_sizes, 'k--', alpha=0.3)

for i, wv in enumerate(wait_values):
    ax.annotate(f'{wv}ms', (avg_waits[i], avg_sizes[i]),
                textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.set_xlabel('Avg Wait Time (ms)')
ax.set_ylabel('Avg Batch Size')
ax.set_title('Latency-Throughput Tradeoff (max_wait_ms sweep)')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, label='max_wait_ms')
plt.tight_layout()
plt.show()

print("Observation: Higher max_wait_ms → larger batches (throughput) but higher latency.")
print("The 'knee' of the curve is often the best operating point.")